**IMPORTANT:** Restart kernel, then run cells in order. Setup (JAVA_HOME + HADOOP_HOME) must run BEFORE SparkSession. After installing duckdb (cell 5), restart kernel again before Q2.

# Module 6 Homework - Yellow November 2025

## Setup: Download data (run once)

Yellow: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet  
Zones: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [1]:
# RUN THIS BEFORE SPARK - required for Parquet write on Windows
import os
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot'
hadoop_home = os.path.join(os.getcwd(), 'hadoop_home', 'hadoop-3.2.1')
os.environ['HADOOP_HOME'] = hadoop_home
if not os.path.exists(os.path.join(hadoop_home, 'bin', 'winutils.exe')):
    print("WARNING: winutils not found. Parquet write may fail. Run hadoop_home install script.")
print("Setup done.")

Setup done.


In [2]:
# Download data (Windows-compatible)
import urllib.request
import os

os.makedirs('data', exist_ok=True)
urllib.request.urlretrieve(
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet',
    'data/yellow_tripdata_2025-11.parquet'
)
urllib.request.urlretrieve(
    'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv',
    'data/taxi_zone_lookup.csv'
)
print('Downloaded.')

Downloaded.


In [3]:
# Install duckdb for parquet read/write (works on Windows ARM64; pyarrow fails to build)
%pip install duckdb -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Q1: Install Spark and PySpark - spark.version

In [4]:
# Set JAVA_HOME for Windows (run before import)
import os
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot'
if 'HADOOP_HOME' not in os.environ:
    os.environ['HADOOP_HOME'] = os.path.join(os.getcwd(), 'hadoop_home', 'hadoop-3.2.1')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
spark = SparkSession.builder.master("local[*]").appName('hw6').getOrCreate()

In [6]:
spark.version

'4.1.1'

## Q2: Yellow Nov 2025 - Repartition to 4, save Parquet, avg file size (MB)

In [7]:
# Q2: Read, repartition to 4, save parquet (DuckDB - works on Windows ARM64; pyarrow fails to build)
import os
import duckdb
os.makedirs('data/pq/yellow/2025/11/', exist_ok=True)
con = duckdb.connect()
con.execute("""
  CREATE TEMP TABLE t AS 
  SELECT *, row_number() OVER () as rn FROM read_parquet('data/yellow_tripdata_2025-11.parquet')
""")
for i in range(4):
    con.execute(f"""
      COPY (SELECT * EXCLUDE (rn) FROM t WHERE (rn-1) % 4 = {i})
      TO 'data/pq/yellow/2025/11/part-{i:05d}.parquet' (FORMAT PARQUET)
    """)
con.close()
# Keep Spark df for Q3-Q6
df_yellow = spark.read.parquet('data/yellow_tripdata_2025-11.parquet')
print('Saved.')

Saved.


In [8]:
# Avg size of parquet files in MB
import glob
files = glob.glob('data/pq/yellow/2025/11/*.parquet')
total_mb = sum(os.path.getsize(f) for f in files) / (1024*1024)
avg_mb = total_mb / len(files) if files else 0
print(f'Files: {len(files)}, Total: {total_mb:.2f} MB, Avg: {avg_mb:.2f} MB')

Files: 4, Total: 87.92 MB, Avg: 21.98 MB


## Q3: Trips on November 15

In [9]:
df_yellow.filter(F.to_date('tpep_pickup_datetime') == '2025-11-15').count()

162604

## Q4: Longest trip in hours

In [10]:
df_yellow \
    .withColumn('duration_sec', F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) \
    .agg(F.max('duration_sec').alias('max_sec')) \
    .withColumn('max_hours', F.col('max_sec') / 3600) \
    .select('max_hours') \
    .collect()[0]['max_hours']

90.64666666666666

## Q5: Spark UI port → **4040**

## Q6: Least frequent pickup zone

In [11]:
df_zones = spark.read.option('header', 'true').csv('data/taxi_zone_lookup.csv')
df_zones.createOrReplaceTempView('zones')

In [13]:
df_yellow.createOrReplaceTempView('yellow')

In [ ]:
spark.sql("""
SELECT z.Zone, COUNT(1) as cnt
FROM yellow y
JOIN zones z ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY cnt ASC
LIMIT 20
""").show()